
# Phase 1 — Lesson 7: Controlled LLM Experiment

## 1. Research Question

Does adding verified product facts and an instruction to avoid unsupported claims improve the factual support of an LLM-generated product description?

## 2. Problem Statement

Large language models can generate fluent and persuasive text, but their outputs may contain factual claims that are not supported by the information provided.

In an e-commerce application, unsupported product claims can mislead customers and damage trust.

This experiment investigates whether providing verified facts together with an explicit grounding instruction improves the factual support of generated product descriptions.

## 3. Hypothesis

Condition B will achieve a higher factual claim support rate than Condition A.

This is a hypothesis to test, not an assumed result.

## 4. Experimental Conditions

### Condition A — Minimal Context

Provide only the product name and ask the model to write a short description.

### Condition B — Verified Facts and Constraint

Provide the verified product name, material, and color. Instruct the model not to invent or imply additional product attributes.

## 5. Variables

- Independent variable: Prompt condition (A or B).
- Primary dependent variable: Factual claim support rate.
- Additional evaluation measures: Number of unsupported claims and description usefulness/completeness.
- Controlled factors: Product, model, generation settings, evaluation rubric, and scoring procedure.
- Unit of analysis: One generated product description.

## 6. Evaluation Criteria

Classify each factual claim as:

- Supported: Directly supported by the verified product facts.
- Unsupported: Not established by the available evidence.
- Contradicted: Conflicts with the verified product facts.

Calculate:

Factual Claim Support Rate =
(Supported factual claims / Total factual claims evaluated) × 100%

Also report the total number of claims and the number of unsupported and contradicted claims.

If no factual claims are made, report the support rate as undefined rather than 100%.

## 7. Experimental Integrity

- Preserve the exact prompts and raw model outputs.
- Record the actual model identifier, date, generation settings, and environment.
- Apply the same evaluation criteria to both conditions.
- Do not fabricate, rewrite, or selectively discard model outputs.
- Distinguish observed results from interpretations and hypotheses.
- Record errors, limitations, and any deviations from the planned procedure.

## 8. Limitations

Condition B changes both the information provided and the instruction given to the model. Therefore, the experiment cannot isolate the effect of verified facts from the effect of the instruction.

A single product and one generation per condition provide limited evidence. Results do not establish universal superiority or guarantee factual reliability in other applications.

## 9. Expected Deliverables

1. Executable Jupyter notebook.
2. Research question, hypothesis, variables, and experimental design.
3. Exact prompts, model details, settings, and environment record.
4. Actual raw outputs from both conditions.
5. Claim-level evaluation rubric and scoring table.
6. Quantitative results and comparison.
7. Interpretation, limitations, and proposed next experiment.
8. Saved experiment data and final reflection.


In [2]:

from datetime import datetime, timezone
import json
import platform
import sys

# Experiment identity
EXPERIMENT_NAME = "lesson_07_product_description_grounding"

# Fictional product with verified facts for this experiment
product = {
    "name": "Everyday Shirt",
    "material": "cotton",
    "color": "blue",
}

# Planned generation settings.
# We will explicitly apply supported settings when calling the API.
GENERATION_CONFIG = {
    "max_output_tokens": 150,
}

# Record the environment and experiment configuration
experiment_metadata = {
    "experiment_name": EXPERIMENT_NAME,
    "started_at_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": sys.version,
    "platform": platform.platform(),
    "product_facts": product,
    "planned_generation_config": GENERATION_CONFIG,
}

print(json.dumps(experiment_metadata, indent=2))



{
  "experiment_name": "lesson_07_product_description_grounding",
  "started_at_utc": "2026-10-27T06:03:10.605137+00:00",
  "python_version": "3.12.12 (main, Dec  3 2025, 10:27:39) [GCC 9.4.0]",
  "platform": "Linux-5.15.167.4-microsoft-standard-WSL2-x86_64-with-glibc2.31",
  "product_facts": {
    "name": "Everyday Shirt",
    "material": "cotton",
    "color": "blue"
  },
  "planned_generation_config": {
    "max_output_tokens": 150
  }
}


In [3]:

# Condition A: Minimal context
prompt_a = """
Write a short product description for a product named "Everyday Shirt".
"""

# Condition B: Verified facts + grounding instruction
prompt_b = f"""
Write a short product description for the product below.

Verified product facts:
- Name: {product["name"]}
- Material: {product["material"]}
- Color: {product["color"]}

Use only the verified facts provided. Do not invent or imply additional
product attributes. If a detail is unknown, leave it out.
"""

# Display the exact prompts that will be sent to the model
print("=" * 60)
print("CONDITION A — MINIMAL CONTEXT")
print("=" * 60)
print(prompt_a)

print("=" * 60)
print("CONDITION B — VERIFIED FACTS + CONSTRAINT")
print("=" * 60)
print(prompt_b)


CONDITION A — MINIMAL CONTEXT

Write a short product description for a product named "Everyday Shirt".

CONDITION B — VERIFIED FACTS + CONSTRAINT

Write a short product description for the product below.

Verified product facts:
- Name: Everyday Shirt
- Material: cotton
- Color: blue

Use only the verified facts provided. Do not invent or imply additional
product attributes. If a detail is unknown, leave it out.



## Before running in Google Colab

1. Open the notebook in Colab.
2. Open **Secrets** in the left sidebar.
3. Add a secret named `GEMINI_API_KEY` and enable notebook access.
4. Install the `google-genai` package if it is not already available.
5. Run the notebook cells in order. The experiment cell sends new API requests.

Never write the API key directly into a notebook cell or share it in outputs.

In [ ]:
%pip install -q google-genai


In [4]:
import os
from google import genai

# In Google Colab, add GEMINI_API_KEY under the Secrets (key) panel
# and enable notebook access. Locally, use the environment variable.
try:
    from google.colab import userdata
except ImportError:
    userdata = None

if userdata is not None:
    api_key = userdata.get("GEMINI_API_KEY")
else:
    api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise RuntimeError(
        "GEMINI_API_KEY is missing. In Colab, add it to Secrets "
        "and enable notebook access. Locally, set it as an environment variable."
    )

client = genai.Client(api_key=api_key)
print("Gemini client initialized:", client is not None)


API key available: True
Gemini client initialized: True


In [5]:

# Final configuration for both experimental conditions
MODEL_NAME = "gemini-3.8-flash"

ACTIVE_GENERATION_CONFIG = {
    "max_output_tokens": 150,
}

# Record the actual planned model and settings
experiment_metadata["model"] = MODEL_NAME
experiment_metadata["generation_config"] = ACTIVE_GENERATION_CONFIG

print("Model:", MODEL_NAME)
print("Generation configuration:")
print(json.dumps(ACTIVE_GENERATION_CONFIG, indent=2))

print("\nMetadata updated:")
print(json.dumps(experiment_metadata, indent=2))


Model: gemini-3.8-flash
Generation configuration:
{
  "max_output_tokens": 150
}

Metadata updated:
{
  "experiment_name": "lesson_07_product_description_grounding",
  "started_at_utc": "2026-10-27T06:03:10.605137+00:00",
  "python_version": "3.12.12 (main, Dec  3 2025, 10:27:39) [GCC 9.4.0]",
  "platform": "Linux-5.15.167.4-microsoft-standard-WSL2-x86_64-with-glibc2.31",
  "product_facts": {
    "name": "Everyday Shirt",
    "material": "cotton",
    "color": "blue"
  },
  "planned_generation_config": {
    "max_output_tokens": 150
  },
  "model": "gemini-3.8-flash",
  "generation_config": {
    "max_output_tokens": 150
  }
}


In [7]:

from datetime import datetime, timezone

# Store the real experiment outputs
experiment_results = {
    "experiment_name": EXPERIMENT_NAME,
    "model_requested": MODEL_NAME,
    "generation_config": ACTIVE_GENERATION_CONFIG.copy(),
    "product_facts": product.copy(),
    "executed_at_utc": datetime.now(timezone.utc).isoformat(),
    "conditions": {},
}

# Run both conditions with the same configuration
for condition_name, prompt in [
    ("A", prompt_a),
    ("B", prompt_b),
]:
    print(f"Running Condition {condition_name}...")

    response = client.interactions.create(
        model=MODEL_NAME,
        input=prompt,
        generation_config=ACTIVE_GENERATION_CONFIG,
    )

    raw_output = response.output_text

    experiment_results["conditions"][condition_name] = {
        "prompt": prompt,
        "raw_output": raw_output,
    }

    print(f"\n--- Condition {condition_name}: Actual model output ---")
    print(raw_output)
    print()

print("Both condition requests completed.")


Running Condition A...

--- Condition A: Actual model output ---
: Classic, timeless, flattering

Running Condition B...

--- Condition B: Actual model output ---
The Everyday Shirt is a blue

Both condition requests completed.


In [8]:
from pathlib import Path
import json

# Create a directory for experiment artifacts
output_dir = Path("data")
output_dir.mkdir(parents=True, exist_ok=True)

# Save the actual outputs and experiment metadata
raw_results_path = output_dir / "lesson_07_raw_results.json"

with raw_results_path.open("w", encoding="utf-8") as file:
    json.dump(experiment_results, file, indent=2, ensure_ascii=False)

print("Raw experiment results saved to:", raw_results_path.resolve())
print("Conditions saved:", list(experiment_results["conditions"].keys()))

# Verify that the saved file can be read
with raw_results_path.open("r", encoding="utf-8") as file:
    saved_results = json.load(file)

assert saved_results["conditions"]["A"]["raw_output"] == \
       experiment_results["conditions"]["A"]["raw_output"]

assert saved_results["conditions"]["B"]["raw_output"] == \
       experiment_results["conditions"]["B"]["raw_output"]

print("Verification passed: both actual outputs were preserved.")

Raw experiment results saved to: /home/tamirat963/genai-foundations-l7/data/lesson_07_raw_results.json
Conditions saved: ['A', 'B']
Verification passed: both actual outputs were preserved.


In [9]:
# Cell 8: Define the claim-level evaluation rubric

evaluation_rubric = {
    "supported": (
        "The claim is directly supported by the verified product facts."
    ),
    "unsupported": (
        "The claim is not established by the verified product facts."
    ),
    "contradicted": (
        "The claim conflicts with the verified product facts."
    ),
}

verified_facts = product.copy()

print("VERIFIED FACTS")
for fact, value in verified_facts.items():
    print(f"- {fact.capitalize()}: {value}")

print("\nEVALUATION RUBRIC")
for category, definition in evaluation_rubric.items():
    print(f"- {category.capitalize()}: {definition}")

VERIFIED FACTS
- Name: Everyday Shirt
- Material: cotton
- Color: blue

EVALUATION RUBRIC
- Supported: The claim is directly supported by the verified product facts.
- Unsupported: The claim is not established by the verified product facts.
- Contradicted: The claim conflicts with the verified product facts.


In [11]:
# Cell 9: Manually evaluate the actual model outputs

claim_evaluations = [
    {
        "condition": "A",
        "claim": "Classic",
        "status": "unsupported",
        "reason": "The verified facts do not establish the product's style.",
    },
    {
        "condition": "A",
        "claim": "timeless",
        "status": "unsupported",
        "reason": "The verified facts provide no evidence for this claim.",
    },
    {
        "condition": "A",
        "claim": "flattering",
        "status": "unsupported",
        "reason": "No evidence about fit or appearance was provided.",
    },
    {
        "condition": "B",
        "claim": "Everyday Shirt",
        "status": "supported",
        "reason": "Matches the verified product name.",
    },
    {
        "condition": "B",
        "claim": "blue",
        "status": "supported",
        "reason": "Matches the verified product color.",
    },
]

# Display the evaluation as a readable table using built-in Python
headers = ["Condition", "Claim", "Status", "Reason"]
rows = [
    [item["condition"], item["claim"], item["status"], item["reason"]]
    for item in claim_evaluations
]

column_widths = [
    max(len(headers[i]), *(len(str(row[i])) for row in rows))
    for i in range(len(headers))
]

def format_row(row):
    return " | ".join(
        str(value).ljust(column_widths[i])
        for i, value in enumerate(row)
    )

print(format_row(headers))
print("-+-".join("-" * width for width in column_widths))

for row in rows:
    print(format_row(row))

print(f"\nTotal claims evaluated: {len(claim_evaluations)}")

Condition | Claim          | Status      | Reason                                                  
----------+----------------+-------------+---------------------------------------------------------
A         | Classic        | unsupported | The verified facts do not establish the product's style.
A         | timeless       | unsupported | The verified facts provide no evidence for this claim.  
A         | flattering     | unsupported | No evidence about fit or appearance was provided.       
B         | Everyday Shirt | supported   | Matches the verified product name.                      
B         | blue           | supported   | Matches the verified product color.                     

Total claims evaluated: 5


In [12]:
# Cell 10: Calculate factual support metrics

condition_metrics = {}

for condition in ["A", "B"]:
    condition_claims = [
        claim for claim in claim_evaluations
        if claim["condition"] == condition
    ]

    supported = sum(
        claim["status"] == "supported"
        for claim in condition_claims
    )
    unsupported = sum(
        claim["status"] == "unsupported"
        for claim in condition_claims
    )
    contradicted = sum(
        claim["status"] == "contradicted"
        for claim in condition_claims
    )
    total = len(condition_claims)

    support_rate = (
        supported / total * 100
        if total > 0
        else None
    )

    condition_metrics[condition] = {
        "total_claims": total,
        "supported": supported,
        "unsupported": unsupported,
        "contradicted": contradicted,
        "support_rate_percent": support_rate,
    }

for condition, metrics in condition_metrics.items():
    print(f"\nCONDITION {condition}")
    print(f"Total claims: {metrics['total_claims']}")
    print(f"Supported: {metrics['supported']}")
    print(f"Unsupported: {metrics['unsupported']}")
    print(f"Contradicted: {metrics['contradicted']}")

    if metrics["support_rate_percent"] is None:
        print("Support rate: Undefined (no claims evaluated)")
    else:
        print(
            f"Factual claim support rate: "
            f"{metrics['support_rate_percent']:.1f}%"
        )

rate_a = condition_metrics["A"]["support_rate_percent"]
rate_b = condition_metrics["B"]["support_rate_percent"]

if rate_a is not None and rate_b is not None:
    difference = rate_b - rate_a
    print(f"\nDifference (B - A): {difference:+.1f} percentage points")


CONDITION A
Total claims: 3
Supported: 0
Unsupported: 3
Contradicted: 0
Factual claim support rate: 0.0%

CONDITION B
Total claims: 2
Supported: 2
Unsupported: 0
Contradicted: 0
Factual claim support rate: 100.0%

Difference (B - A): +100.0 percentage points


In [13]:
# Cell 11: Evaluate completeness independently of factual support

completeness_rubric = {
    0: "No useful product description",
    1: "Minimal or incomplete description",
    2: "Understandable description with limited detail",
    3: "Clear, useful description using available product facts",
}

completeness_evaluations = {
    "A": {
        "score": 1,
        "reason": (
            "Uses generic promotional words but does not identify "
            "the product or communicate its verified attributes."
        ),
    },
    "B": {
        "score": 1,
        "reason": (
            "Identifies the product and its color, but the sentence "
            "is incomplete and omits the verified material."
        ),
    },
}

for condition, evaluation in completeness_evaluations.items():
    print(f"\nCONDITION {condition}")
    print("Actual output:", experiment_results["conditions"][condition]["raw_output"])
    print(f"Completeness score: {evaluation['score']}/3")
    print("Reason:", evaluation["reason"])

print("\nCOMPLETENESS RUBRIC")
for score, definition in completeness_rubric.items():
    print(f"{score}: {definition}")


CONDITION A
Actual output: : Classic, timeless, flattering
Completeness score: 1/3
Reason: Uses generic promotional words but does not identify the product or communicate its verified attributes.

CONDITION B
Actual output: The Everyday Shirt is a blue
Completeness score: 1/3
Reason: Identifies the product and its color, but the sentence is incomplete and omits the verified material.

COMPLETENESS RUBRIC
0: No useful product description
1: Minimal or incomplete description
2: Understandable description with limited detail
3: Clear, useful description using available product facts


In [14]:
# Cell 12: Consolidate the experiment metrics

final_comparison = {}

for condition in ["A", "B"]:
    final_comparison[condition] = {
        **condition_metrics[condition],
        "completeness_score": completeness_evaluations[condition]["score"],
        "completeness_reason": completeness_evaluations[condition]["reason"],
    }

headers = [
    "Condition",
    "Support rate",
    "Supported",
    "Unsupported",
    "Contradicted",
    "Completeness",
]

rows = []

for condition, metrics in final_comparison.items():
    rows.append([
        condition,
        f"{metrics['support_rate_percent']:.1f}%",
        metrics["supported"],
        metrics["unsupported"],
        metrics["contradicted"],
        f"{metrics['completeness_score']}/3",
    ])

widths = [
    max(len(headers[i]), *(len(str(row[i])) for row in rows))
    for i in range(len(headers))
]

def print_table_row(row):
    print(" | ".join(
        str(value).ljust(widths[i])
        for i, value in enumerate(row)
    ))

print_table_row(headers)
print("-+-".join("-" * width for width in widths))

for row in rows:
    print_table_row(row)

print("\nOBSERVED FINDINGS")
print(
    "Factual support-rate difference: "
    f"{condition_metrics['B']['support_rate_percent'] - condition_metrics['A']['support_rate_percent']:+.1f} "
    "percentage points."
)
print(
    "Completeness-score difference: "
    f"{completeness_evaluations['B']['score'] - completeness_evaluations['A']['score']:+d} points."
)
print(
    "Interpretation: Condition B had better factual support in this run, "
    "but neither output was a complete, useful product description."
)

Condition | Support rate | Supported | Unsupported | Contradicted | Completeness
----------+--------------+-----------+-------------+--------------+-------------
A         | 0.0%         | 0         | 3           | 0            | 1/3         
B         | 100.0%       | 2         | 0           | 0            | 1/3         

OBSERVED FINDINGS
Factual support-rate difference: +100.0 percentage points.
Completeness-score difference: +0 points.
Interpretation: Condition B had better factual support in this run, but neither output was a complete, useful product description.


In [15]:
# Cell 13: Save evaluation results alongside the raw model outputs

evaluation_results = {
    "experiment_name": EXPERIMENT_NAME,
    "research_question": (
        "Does adding verified product facts and an instruction to avoid "
        "unsupported claims improve factual support in an LLM-generated "
        "product description?"
    ),
    "hypothesis": (
        "Condition B will have a higher factual claim support rate than A."
    ),
    "verified_product_facts": product.copy(),
    "rubric": evaluation_rubric,
    "claim_evaluations": claim_evaluations,
    "condition_metrics": condition_metrics,
    "completeness_rubric": completeness_rubric,
    "completeness_evaluations": completeness_evaluations,
    "observed_conclusion": (
        "In this single run per condition, B had a higher factual claim "
        "support rate than A, while both outputs received the same "
        "completeness score. The result is preliminary and does not "
        "establish general model behavior."
    ),
    "limitations": [
        "Only one generation was collected per condition.",
        "The conditions differ in both supplied facts and instructions.",
        "Claim classification and completeness scoring involve human judgment.",
        "The outputs were very short and neither was a complete description.",
        "The observed percentages do not establish statistical significance.",
    ],
}

evaluation_path = output_dir / "lesson_07_evaluation_results.json"

with evaluation_path.open("w", encoding="utf-8") as file:
    json.dump(evaluation_results, file, indent=2, ensure_ascii=False)

# Verify the saved evaluation
with evaluation_path.open("r", encoding="utf-8") as file:
    verified_evaluation = json.load(file)

assert len(verified_evaluation["claim_evaluations"]) == 5
assert verified_evaluation["condition_metrics"]["A"]["support_rate_percent"] == 0.0
assert verified_evaluation["condition_metrics"]["B"]["support_rate_percent"] == 100.0

print("Evaluation results saved to:", evaluation_path.resolve())
print("Claim records:", len(verified_evaluation["claim_evaluations"]))
print("Condition A support rate:",
      verified_evaluation["condition_metrics"]["A"]["support_rate_percent"], "%")
print("Condition B support rate:",
      verified_evaluation["condition_metrics"]["B"]["support_rate_percent"], "%")
print("Verification passed.")

Evaluation results saved to: /home/tamirat963/genai-foundations-l7/data/lesson_07_evaluation_results.json
Claim records: 5
Condition A support rate: 0.0 %
Condition B support rate: 100.0 %
Verification passed.


In [16]:
# Cell 14: Generate the final experiment report

a = final_comparison["A"]
b = final_comparison["B"]

report = f"""
# Phase 1 — Lesson 7: Controlled LLM Experiment

## Research question
Does adding verified product facts and an instruction to avoid unsupported
claims improve factual support in an LLM-generated product description?

## Hypothesis
Condition B will have a higher factual claim support rate than Condition A.

## Experimental design
- Condition A: Minimal prompt containing the product name.
- Condition B: Verified product facts plus an instruction not to invent
  or imply additional attributes.
- Product: Everyday Shirt.
- Verified facts: name = Everyday Shirt; material = cotton; color = blue.
- Generations: one per condition.
- Model requested: {MODEL_NAME}.

## Actual model outputs

### Condition A
{experiment_results["conditions"]["A"]["raw_output"]}

### Condition B
{experiment_results["conditions"]["B"]["raw_output"]}

## Results

| Metric | Condition A | Condition B |
|---|---:|---:|
| Claims evaluated | {a['total_claims']} | {b['total_claims']} |
| Supported claims | {a['supported']} | {b['supported']} |
| Unsupported claims | {a['unsupported']} | {b['unsupported']} |
| Contradicted claims | {a['contradicted']} | {b['contradicted']} |
| Factual claim support rate | {a['support_rate_percent']:.1f}% | {b['support_rate_percent']:.1f}% |
| Completeness score | {a['completeness_score']}/3 | {b['completeness_score']}/3 |

Observed support-rate difference (B - A):
{b['support_rate_percent'] - a['support_rate_percent']:+.1f} percentage points.

Observed completeness-score difference (B - A):
{b['completeness_score'] - a['completeness_score']:+d} points.

## Interpretation
The observed results are consistent with the hypothesis: Condition B had a
higher factual claim support rate in this run. However, its description was
still incomplete. Condition A used unsupported promotional language, while
Condition B identified the product and its color but did not finish the
sentence or mention the material.

## Limitations
1. Only one generation was collected per condition.
2. Condition B changes both the information provided and the instructions,
   so their individual effects cannot be isolated.
3. Claim classification and completeness scoring involve human judgment.
4. The sample is too small to support a broad claim about model behavior.
5. The observed 100% support rate for Condition B does not imply that it will
   achieve 100% on other products or repeated generations.

## Next experiment
Repeat the experiment across multiple products and multiple generations.
Use separate conditions to distinguish the effect of verified facts from
the effect of the instruction. Add a completeness requirement and measure
both factual support and usefulness.

## Conclusion
In this limited experiment, supplying verified facts with an explicit
grounding instruction coincided with better factual claim support, but did
not improve completeness. Further controlled repetitions are needed before
drawing a general conclusion.
"""

report_path = output_dir / "lesson_07_final_report.md"
report_path.write_text(report.strip() + "\n", encoding="utf-8")

print("Final report saved to:", report_path.resolve())
print("\n" + report)

Final report saved to: /home/tamirat963/genai-foundations-l7/data/lesson_07_final_report.md


# Phase 1 — Lesson 7: Controlled LLM Experiment

## Research question
Does adding verified product facts and an instruction to avoid unsupported
claims improve factual support in an LLM-generated product description?

## Hypothesis
Condition B will have a higher factual claim support rate than Condition A.

## Experimental design
- Condition A: Minimal prompt containing the product name.
- Condition B: Verified product facts plus an instruction not to invent
  or imply additional attributes.
- Product: Everyday Shirt.
- Verified facts: name = Everyday Shirt; material = cotton; color = blue.
- Generations: one per condition.
- Model requested: gemini-3.8-flash.

## Actual model outputs

### Condition A
: Classic, timeless, flattering

### Condition B
The Everyday Shirt is a blue

## Results

| Metric | Condition A | Condition B |
|---|---:|---:|
| Claims evaluated | 3 | 2 |
| Supported claims |